# **Probabilidades con modelo Naive Bayes**
### **AUTOR:** HADSON PAREDES
### **CASO:** Sistema de clasificación binaria de correos electrónicos.

[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Networking: Linkedin](https://img.shields.io/badge/LinkedIn-Hadson%20Paredes-blue?logo=linkedin&style=flat)](https://www.linkedin.com/in/hadson-paredes/) 
[![Networking: Facebook](https://img.shields.io/badge/Facebook-Hadson%20Paredes%20Cordova-Gree?logo=facebook&style=flat)](https://www.facebook.com/hadson.paredescordova/) 
[![Networking: X](https://img.shields.io/badge/Hadson%20Paredes-black?logo=x&style=flat)](https://x.com/hadson_paredes)

En este **_caso completo_ aplicaremos (integramos) los conceptos de teoría de probabilidad con un modelo Naive Bayes** aplicado a la simulación para la detección de **_correos Spam_**.

## 1. Objetivo principal

El objetivo principal es construir un **sistema de clasificación binaria de correos electrónicos** para determinar si un mensaje entrante es **Spam** o **Ham** (correo legítimo) basándose en la presencia de palabras clave específicas (`Contiene_Gratis` y `Contiene_Urgente`). El análisis se aplica a un conjunto de datos histórico (`dataset_spam.csv`) de 10 observaciones (registros) para calcular las distribuciones de probabilidad requeridas por un modelo **Naive Bayes**, fundamentando la toma de decisiones automatizada bajo el supuesto de independencia condicional.

> Recordemos que el modelo o teorema de Naive Bayes tamibién nos referimos al teorema clasificador Bayesiano Ingenuo.

## 2. Fórmulas Matemáticas a Considerar

Para el modelado probabilístico y el funcionamiento del clasificador se aplican tres conceptos fundamentales de la teoría de probabilidad, considerando el Teorema de Bayes: 

* **Probabilidad Conjunta:** Mide la probabilidad de que dos eventos ocurran al mismo tiempo.
* **Probabilidad Marginal:** La probabilidad de que ocurra un evento simple sin considerar otras variables. Se obtiene sumando las probabilidades conjuntas.
* **Probabilidad Condicional:** La probabilidad de un evento dado que otro ya ha ocurrido.
* **Clasificador Naive Bayes (Inferencia):** Asume que las palabras clave ($X_1, X_2$) son condicionalmente independientes dado el estado del correo ($Y$). Para clasificar un correo que contiene ambas palabras, buscamos la clase que maximice el numerador del Teorema de Bayes:

> Revisa las formulas matemáticas en el punto [2. Fórmulas Matemáticas del Teorema de Bayes (Naive Bayes)](README.md)

## 3. El Conjunto de Datos

El conjunto de datos (dataset) simula un listado de correos electrónicos clasificados bajo tres variables binarias:

* `Contiene_Gratis`: El correo contiene la palabra "Gratis" (1: Sí, 0: No).
* `Contiene_Urgente`: El correo contiene la palabra "Urgente" (1: Sí, 0: No).
* `Es_Spam`: La **etiqueta objetivo** (1: Spam, 0: Correo Legítimo / Ham).

Estructura del archivo llamado `dataset_spam.csv`:

```text
Contiene_Gratis,Contiene_Urgente,Es_Spam
1,1,1
1,0,1
0,1,1
0,0,0
1,0,0
0,1,0
1,1,1
0,0,0
1,1,0
0,1,1
```

## 4. Estructura Teórica y desarrollo de Probabilidades

En términos probabilísticos, las distribuciones se calculan de la siguiente manera:

* **Probabilidad Conjunta $P(A, B)$:** Es la probabilidad de que dos o más eventos ocurran simultáneamente. Se calcula dividiendo la frecuencia del caso específico entre el total de muestras.

* **Probabilidad Marginal $P(A)$:** Es la probabilidad de un evento simple, sin importar el resultado de otras variables. Se obtiene sumando las probabilidades conjuntas de la variable de interés.

* **Probabilidad Condicional $P(A \vert{} B)$:** La probabilidad de que ocurra $A$ dado que ya ocurrió el evento $B$. Se define matemáticamente como:

$$P(A \vert{} B) = \frac{P(A, B)}{P(B)}$$

### 4.1. El Modelo Naive Bayes

Un clasificador **Naive Bayes** aplica el Teorema de Bayes asumiendo que las variables predictoras son **independientes entre sí**, dado el conocimiento de la clase. Para predecir si un mensaje nuevo es _Spam_ dado que contiene ciertas palabras clave, calcula:

$$P(\text{Spam} \vert{} \text{Gratis}, \text{Urgente}) \propto P(\text{Spam}) \times P(\text{Gratis} \vert{} \text{Spam}) \times P(\text{Urgente} \vert{} \text{Spam})$$

### 4.2. Desarrollo de la predicción según el caso de estudio

Calcular la estadística descriptiva y simular un motor de clasificación **Naive Bayes** para el **sistema de clasificación binaria de correos electrónicos** para determinar si un mensaje entrante es **Spam** o **Ham** (correo legítimo) basándose en la presencia de palabras clave específicas (`Contiene_Gratis` y `Contiene_Urgente`).

1. Importar librerías necesarias, Crear/Cargar el dataset.csv directamente y mostrar el total de observaciones (registros)

In [ ]:
# Importar librerías necesarias
import pandas as pd
#import numpy as np

# 1. Crear/Cargar el dataset_spam.csv directamente
data = {
    'Contiene_Gratis':   [1, 1, 0, 0, 1, 0, 1, 0, 1, 0],
    'Contiene_Urgente':  [1, 0, 1, 0, 0, 1, 1, 0, 1, 1],
    'Es_Spam':            [1, 1, 1, 0, 0, 0, 1, 0, 0, 1]
}
df = pd.DataFrame(data)
df.to_csv('dataset_spam.csv', index=False)

print("--- DATASET CARGADO ---")
print(df, "\n")

total_observaciones = len(df)
print(f"Total de observaciones en el dataset: {total_observaciones}\n")

--- DATASET CARGADO ---
   Contiene_Gratis  Contiene_Urgente  Es_Spam
0                1                 1        1
1                1                 0        1
2                0                 1        1
3                0                 0        0
4                1                 0        0
5                0                 1        0
6                1                 1        1
7                0                 0        0
8                1                 1        0
9                0                 1        1 

Total de observaciones en el dataset: 10



2. PROBABILIDAD CONJUNTA

In [ ]:
print("--- 1. DISTRIBUCIÓN DE PROBABILIDAD CONJUNTA ---")
# Frecuencia relativa de que ocurra Contiene_Gratis y Es_Spam simultáneamente
tabla_conjunta = pd.crosstab(df['Contiene_Gratis'], df['Es_Spam'], normalize='all')
print(tabla_conjunta, "\n")

# Ejemplo específico de la tabla
p_gratis_y_spam = tabla_conjunta.loc[1, 1]
print(f"P(Contiene_Gratis=1, Es_Spam=1) = {p_gratis_y_spam:.2f}\n")

# Ejemplo adicional: Probabilidad de que no contenga gratis y sea ham
p_no_gratis_y_ham = tabla_conjunta.loc[0, 0]
print(f"P(Contiene_Gratis=0, Es_Spam=0) = {p_no_gratis_y_ham:.2f}\n")

--- 1. DISTRIBUCIÓN DE PROBABILIDAD CONJUNTA ---
Es_Spam            0    1
Contiene_Gratis          
0                0.3  0.2
1                0.2  0.3 

P(+Contiene_Gratis=1, Es_Spam=1) = 0.30

P(Contiene_Gratis=0, Es_Spam=0) = 0.30



3. PROBABILIDAD MARGINAL

In [19]:
print("--- 2. DISTRIBUCIÓN DE PROBABILIDAD MARGINAL ---")
# Probabilidad individual de las clases (Spam vs Legítimo)
p_marginal_spam = df['Es_Spam'].value_counts(normalize=True)
print("Marginal de la variable Objetivo (Es_Spam):")
print(p_marginal_spam, "\n")

p_spam = p_marginal_spam[1]
p_ham = p_marginal_spam[0]

print(f"Probabilidad de que sea Spam: {p_spam:.4f}")
print(f"Probabilidad de que sea Ham (Legítimo): {p_ham:.4f}")

--- 2. DISTRIBUCIÓN DE PROBABILIDAD MARGINAL ---
Marginal de la variable Objetivo (Es_Spam):
Es_Spam
1    0.5
0    0.5
Name: proportion, dtype: float64 

Probabilidad de que sea Spam: 0.5000
Probabilidad de que sea Ham (Legítimo): 0.5000


4. PROBABILIDAD CONDICIONAL

In [20]:
print("--- 3. DISTRIBUCIÓN DE PROBABILIDAD CONDICIONAL ---")
# P(Palabra | Clase) -> Normalizando por columnas (la clase dada)
tabla_condicional = pd.crosstab(df['Contiene_Gratis'], df['Es_Spam'], normalize='columns')
print("P(Contiene_Gratis | Es_Spam):")
print(tabla_condicional, "\n")

# Probabilidades específicas condicionales necesarias para Naive Bayes
p_gratis_dado_spam = tabla_condicional.loc[1, 1]
p_gratis_dado_ham = tabla_condicional.loc[1, 0]

# Repetimos para la segunda variable ('Contiene_Urgente')
tabla_condicional_urgente = pd.crosstab(df['Contiene_Urgente'], df['Es_Spam'], normalize='columns')
p_urgente_dado_spam = tabla_condicional_urgente.loc[1, 1]
p_urgente_dado_ham = tabla_condicional_urgente.loc[1, 0]

--- 3. DISTRIBUCIÓN DE PROBABILIDAD CONDICIONAL ---
P(Contiene_Gratis | Es_Spam):
Es_Spam            0    1
Contiene_Gratis          
0                0.6  0.4
1                0.4  0.6 



5. MODELO PROBABILÍSTICO ESTILO NAIVE BAYES

In [25]:
print("--- 4. CLASIFICADOR NAIVE BAYES (Predicción de Correo Nuevo) ---")
print("Imagina un correo nuevo que: Contiene_Gratis=1 y Contiene_Urgente=1")

# Aplicando el supuesto de independencia "ingenuo":
# Posterior No Normalizado para Spam = P(Spam) * P(Gratis=1|Spam) * P(Urgente=1|Spam)
print(f"spam: {p_spam:.4f}  gratis: {p_gratis_dado_spam:.4f} urgente: {p_urgente_dado_spam:.4f}")
print(f"-- valores no normalizados --")
posterior_spam_no_norm = p_spam * p_gratis_dado_spam * p_urgente_dado_spam

# Posterior No Normalizado para Ham = P(Ham) * P(Gratis=1|Ham) * P(Urgente=1|Ham)
posterior_ham_no_norm = p_ham * p_gratis_dado_ham * p_urgente_dado_ham

# Normalización para obtener probabilidades reales que sumen 1
evidencia = posterior_spam_no_norm + posterior_ham_no_norm
p_final_spam = posterior_spam_no_norm / evidencia
p_final_ham = posterior_ham_no_norm / evidencia

print(f"Probabilidad de que sea Spam: {p_final_spam:.4f}")
print(f"Probabilidad de que sea Ham (Legítimo): {p_final_ham:.4f}")

# Decisión final del Clasificador
if p_final_spam > p_final_ham:
    print("Resultado de la clasificación: El correo es asignado a SPAM")
else:
    print("Resultado de la clasificación: El correo es asignado a HAM (Legítimo)")

--- 4. CLASIFICADOR NAIVE BAYES (Predicción de Correo Nuevo) ---
Imagina un correo nuevo que: Contiene_Gratis=1 y Contiene_Urgente=1
spam: 0.5000  gratis: 0.6000 urgente: 0.8000
-- valores no normalizados --
Probabilidad de que sea Spam: 0.7500
Probabilidad de que sea Ham (Legítimo): 0.2500
Resultado de la clasificación: El correo es asignado a SPAM


### 4.3. Explicación breve de los resultados del código desarrollado

1. El código **genera la distribución conjunta** calculando qué fracción del total corresponde a cada celda de cruce.
2. Extrae las **probabilidades marginales**, mostrando que, por ejemplo, el 50% de nuestros correos históricos son spam.
3. Encuentra las **probabilidades condicionales** que responden preguntas como: _"Sabiendo que un correo es spam, ¿cuál es la probabilidad de que contenga la palabra 'Gratis'?"_.
4. Finalmente, realiza la inferencia **Naive Bayes** multiplicando la probabilidad previa de la clase por las verosimilitudes de cada palabra independiente, concluyendo de forma automatizada sobre la naturaleza del mensaje.

## 5. Conclusiones

* **Eficiencia del Enfoque Ingenuo:** A pesar de asumir una independencia que rara vez ocurre en el lenguaje real (las palabras suelen correlacionarse), Naive Bayes provee una línea base altamente efectiva, rápida y computacionalmente económica.
* **Fundamento en Datos:** Las distribuciones condicionales permiten segmentar el impacto individual de cada variable. Por ejemplo, si una palabra tiene alta frecuencia en la columna de **Spam** pero baja en la de **Ham**, se convierte en un fuerte predictor positivo.
* **Sensibilidad al Contexto:** El cálculo de la probabilidad posterior equilibra la probabilidad previa (qué tan común es recibir Spam en general) con la evidencia actual (el contenido del mensaje), reduciendo falsos positivos automáticos.


## 6. Próximos Pasos

Para evolucionar este prototipo hacia un entorno productivo y robusto, se sugieren las siguientes acciones:

* **Implementar Suavizado de Laplace (Laplace Smoothing):** Si un correo nuevo contiene una palabra que nunca apareció en los datos de entrenamiento para una clase específica, su probabilidad condicional será $0$, anulando todo el producto multiplicativo. El suavizado añade un pequeño valor al conteo para evitar este fallo masivo.
* **Escalar a Texto Real (NLP):** Sustituir las banderas binarias manuales por técnicas de Procesamiento de Lenguaje Natural como CountVectorizer o TF-IDF de la librería scikit-learn para mapear vocabularios de miles de palabras.
* **Métricas de Evaluación:** Implementar matrices de confusión, cálculo de precisión, Recall y puntuación F1-Score utilizando conjuntos de datos divididos en entrenamiento (Train) y validación (Test) para medir el rendimiento real del clasificador.

<h4 align="center"> Publicaciones en mis redes sociales y repositorio GitHub</h4>

<div align="center">
  <h3>Sígueme en mis redes sociales</h3>
  <a href="https://github.com/devhadson">
    <img src="https://img.shields.io/badge/GitHub-devhadson-black?logo=GitHub&style=flat" target="_blank" alt="GitHub">
  </a>
  <a href="https://www.linkedin.com/in/hadson-paredes/">
    <img src="https://img.shields.io/badge/LinkedIn-Hadson%20Paredes-blue?logo=linkedin&style=flat" target="_blank" alt="LinkedIn">
  </a>
  <a href="https://www.facebook.com/hadson.paredescordova/">
    <img src="https://img.shields.io/badge/Facebook-Hadson%20Paredes%20Cordova-Gree?logo=facebook&style=flat" target="_blank" alt="Facebook">
  </a>
  <a href="https://x.com/hadson_paredes">
    <img src="https://img.shields.io/badge/Hadson%20Paredes-black?logo=x&style=flat" target="_blank" alt="X">
  </a>
    <a href="https://blog.hadsonpar.com/">
    <img src="https://img.shields.io/badge/Blog-blog.hadsonpar-orange?logo=linkedin&style=flat" target="_blank" alt="LinkedIn">
  </a>
</div>